In [1]:
import requests
import pandas as pd

artistas = [
    "Madonna",
    "Whitney Houston",
    "Donna Summer",
    "Cyndi Lauper",
    "Kate Bush",
    "Gloria Gaynor",
    "Bonnie Tyler",
    "Nina Simone",
    "Mercedes Sosa",
    "Rocío Dúrcal"
]


def conseguir_canciones(nombre_artista, limit=50):
    url = "https://api.deezer.com/search"

    params = {
        "q": f'artist:"{nombre_artista}"',
        "limit": limit
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        datos = response.json()

    except Exception as e:
        print(f"Error buscando a {nombre_artista}: {e}")
        return []

    canciones = datos.get("data", [])
    lista_canciones = []

    for cancion in canciones:
        try:
            track_id = cancion["id"]

            track_response = requests.get(
                f"https://api.deezer.com/track/{track_id}"
            )
            track_response.raise_for_status()
            track = track_response.json()

            album_id = track["album"]["id"]

            album_response = requests.get(
                f"https://api.deezer.com/album/{album_id}"
            )
            album_response.raise_for_status()
            album = album_response.json()

            genero = None
            id_genero = album.get("genre_id")

            if album.get("genres") and album["genres"].get("data"):
                genero = album["genres"]["data"][0].get("name")

            if len(track.get("contributors", [])) > 1:
                tipo = "colaboracion"
            else:
                tipo = album.get("record_type", "track")

            if track.get("release_date"):
                anio_lanzamiento = track.get("release_date")[:4]
            else:
                anio_lanzamiento = None

            lista_canciones.append({
                "id_artista": track["artist"]["id"],
                "nombre_artista": track["artist"]["name"],
                "titulo_cancion": track["title"],
                "titulo_album": track["album"]["title"],
                "tipo": tipo,
                "anio_lanzamiento": anio_lanzamiento,
                "genero": genero,
                "id_genero": id_genero
            })

        except Exception as e:
            print(f"Error procesando una canción de {nombre_artista}: {e}")

    return lista_canciones


lista_total_canciones = []

for artista in artistas:
    print(f"Buscando canciones de {artista}...")

    canciones_artista = conseguir_canciones(artista, limit=50)

    lista_total_canciones.extend(canciones_artista)


df_deezer = pd.DataFrame(lista_total_canciones)

df_deezer.to_csv(
    "artistas_completo_deezer_Nati.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV Deezer guardado correctamente")
print(df_deezer.shape)
df_deezer.head()

Buscando canciones de Madonna...
Buscando canciones de Whitney Houston...
Buscando canciones de Donna Summer...
Buscando canciones de Cyndi Lauper...
Buscando canciones de Kate Bush...
Buscando canciones de Gloria Gaynor...
Buscando canciones de Bonnie Tyler...
Buscando canciones de Nina Simone...
Buscando canciones de Mercedes Sosa...
Buscando canciones de Rocío Dúrcal...
CSV Deezer guardado correctamente
(453, 8)


,id_artista,nombre_artista,titulo_cancion,titulo_album,tipo,anio_lanzamiento,genero,id_genero
0,655248,Les artistes des Antilles,Madiana,"Les Antilles, vol. 1 : Cuba, Haiti, La Guadelo...",album,1999,Pop,132
1,1173980,"Doug Walker, Steel Drum Artist",Old Macdonald,"Caribbean Kids Collection, Vol 1",album,2011,Niños,95
2,290,Madonna,Like a Prayer,Like a Prayer,album,2005,Pop,132
3,4781,Joe Bonamassa,To Know You Is To Love You,B.B. King's Blues Summit 100,colaboracion,2026,Rock,152
4,812837,Dolapdere Big Gang,Karma Chameleon,Art-Ist,colaboracion,2010,NaN,-1


In [2]:
import requests
import pandas as pd

BASE_URL = "http://ws.audioscrobbler.com/2.0/"
API_KEY = "f9cc8c962420faeaec7b07fb183f4974"

artistas = [
    "Madonna",
    "Whitney Houston",
    "Donna Summer",
    "Cyndi Lauper",
    "Kate Bush",
    "Gloria Gaynor",
    "Bonnie Tyler",
    "Nina Simone",
    "Mercedes Sosa",
    "Rocío Dúrcal"
]


def buscar_artista(nombre_artista):
    params = {
        "method": "artist.getInfo",
        "artist": nombre_artista,
        "api_key": API_KEY,
        "format": "json",
        "lang": "es"
    }

    try:
        response = requests.get(BASE_URL, params=params)
        response.raise_for_status()
        datos = response.json()

        artista = datos["artist"]

        nombre_artista_lfm = artista["name"]
        biografia = artista["bio"]["summary"]
        listeners = artista["stats"]["listeners"]
        playcount = artista["stats"]["playcount"]

        similares = artista["similar"]["artist"]
        artistas_similares = []

        for artista_similar in similares:
            artistas_similares.append(artista_similar["name"])

        return {
            "nombre_artista": nombre_artista_lfm,
            "biografia": biografia,
            "listeners": listeners,
            "playcount": playcount,
            "artistas_similares": ", ".join(artistas_similares)
        }

    except Exception as e:
        print(f"Error al buscar {nombre_artista}: {e}")

        return {
            "nombre_artista": nombre_artista,
            "biografia": None,
            "listeners": None,
            "playcount": None,
            "artistas_similares": None
        }


lista_lastfm = []

for artista in artistas:
    print(f"Buscando información de {artista}...")

    datos_artista = buscar_artista(artista)

    lista_lastfm.append(datos_artista)


df_lfm = pd.DataFrame(lista_lastfm)

df_lfm.to_csv(
    "artistas_completo_LFM_Nati.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV Last.fm guardado correctamente")
print(df_lfm.shape)
df_lfm.head()

Buscando información de Madonna...
Buscando información de Whitney Houston...
Buscando información de Donna Summer...
Buscando información de Cyndi Lauper...
Buscando información de Kate Bush...
Buscando información de Gloria Gaynor...
Buscando información de Bonnie Tyler...
Buscando información de Nina Simone...
Buscando información de Mercedes Sosa...
Buscando información de Rocío Dúrcal...
CSV Last.fm guardado correctamente
(10, 5)


,nombre_artista,biografia,listeners,playcount,artistas_similares
0,Madonna,"Madonna Louise Veronica Ciccone (Bay City, Míc...",5707724,371493131,"Kylie Minogue, Janet Jackson, Britney Spears, ..."
1,Whitney Houston,"Whitney Elizabeth Houston (Newark, 9 de agosto...",3263420,63618920,"Toni Braxton, Mariah Carey, Céline Dion, Janet..."
2,Donna Summer,"LaDonna Andrea Gaines, (Boston, Massachusetts,...",1645201,24858213,"Diana Ross, Janet Jackson, Kylie Minogue, Cher..."
3,Cyndi Lauper,"Cynthia Ann Stephanie Lauper, más conocida com...",3042711,38868049,"Laura Branigan, Cher, The Bangles, Tina Turner..."
4,Kate Bush,Kate Bush (nacida Catherine Bush el 30 de juli...,3495548,147523239,"Björk, Fiona Apple, Tori Amos, PJ Harvey, Joni..."
